# Text Extraction

In [ ]:
REPO_URL = "https://github.com/Govindkm/tcs-ai-club-hackathon-prompt-pioneers.git"
BRANCH = "develop"
PROJECT_DIR = "/content/tcs-ai-club-hackathon-prompt-pioneers"

import os

if not os.path.isdir(PROJECT_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {PROJECT_DIR}
else:
    print("Repository already cloned - pulling latest changes.")
    !git -C {PROJECT_DIR} pull

%cd {PROJECT_DIR}

## 1. Install Python dependencies

In [ ]:
!pip install -q -r requirements.txt

## 2. Install and start Ollama (text + vision models)

`zstd` is required by the Ollama installer on Colab's base image.

In [ ]:
!sudo apt-get update -qq && sudo apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import os
import subprocess
import time

import requests

os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
ollama_process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(30):
    try:
        requests.get("http://127.0.0.1:11434", timeout=2)
        print("Ollama server is up.")
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    raise RuntimeError("Ollama server did not start in time - check the logs and retry.")

MODEL_NAME = "llama3.2:3b"       # keep in sync with OLLAMA_MODEL_ID in the .env cell below
VISION_MODEL_NAME = "minicpm-v"  # keep in sync with OLLAMA_VISION_MODEL in the .env cell below
!ollama pull {MODEL_NAME}
!ollama pull {VISION_MODEL_NAME}

## 3. Get Text From Files

In [ ]:
from google.colab import files
from src.ingestion.document_reader import extract_text

uploaded = files.upload()
filename = next(iter(uploaded))

text = extract_text(filename, uploaded[filename])
print(text)

## 4. Shutdown (run when you're done)

In [ ]:
for proc in [tunnel_process, ollama_process]:
    proc.terminate()
print("Tunnel and Ollama all stopped.")